# Project Objective


The goal of this project is to develop a **predictive model capable of estimating the probability of default for monthly billing charges issued to customers**.

Default is defined as a payment made **5 or more days late relative to the due date**.

The model must generate a default probability for each record in the payments dataset, which represents the company's most recent billing charges.

For model development, several information sources about customers are available:

* **Payment history**, containing information about payment behavior;
* **Registration and profile data** for customers;
* **Monthly information**, such as income and number of employees;
* Other historical information that may help identify patterns associated with default.

The solution should consider aspects such as:

* data treatment and quality;
* feature creation and selection;
* proper separation between train, validation, and test;
* model performance evaluation;
* results interpretation;
* comparison with an existing decision rule (*proxy*).

The final output must contain **only the default probability for each billing charge**, with values between `0` and `1`.

It is not necessary to convert these probabilities into a binary classification of default or non-default.

> **Note:** the default definition used in this project — **5 or more days late** — is adopted as the rule for target construction and model evaluation.


# Database


The project provides **three datasets** containing registration, behavioral, and financial information about customers. The datasets were extracted from a billing system and represent a realistic operating scenario.

The tables are mainly related through:

* **ID_CLIENTE**: uniquely identifies each customer;
* **SAFRA_REF**: represents the billing reference period.

Available datasets

| Dataset     | Description                                                                 | Granularity        |
| ----------- | --------------------------------------------------------------------------- | ------------------ |
| cadastral   | Registration information, such as registration date, size, ZIP code, and email domain. | Customer           |
| info        | Monthly information, such as income and number of employees.                | Customer × month   |
| pagamentos  | Most recent billing charges, used to generate model training and predictions. | Customer × charge  |

### Role of each dataset

The **cadastral** dataset contains more static information about customers, while **info** allows tracking characteristics that may change over time.

The **pagamentos** dataset is used for model development. Because it contains due dates and payment dates, it is possible to identify which billing charges were in default according to the rule defined in the project.


# Cadastral


The **cadastral** dataset gathers identification and registration characteristics of customers. Each record represents a unique customer, identified by ID_CLIENTE.

The dataset has **1,315 records and 8 variables**.

During initial exploration, some points of attention were observed, mainly the presence of missing values and inconsistencies in the DDD variable, which contains records with numbers and characters.

Data Dictionary

| Variable            | Description                           | Type                | Notes                                           |
| ------------------- | ------------------------------------- | ------------------- | ----------------------------------------------- |
| ID_CLIENTE          | Unique customer identifier            | Integer             | Primary key                                     |
| DATA_CADASTRO       | Date the customer was registered      | Date                | No null values                                  |
| DDD                 | Customer phone area code              | Text/Numeric        | ~18% null and inconsistent values               |
| FLAG_PF             | Individual person indicator           | Binary              | ~95% null                                       |
| SEGMENTO_INDUSTRIAL | Customer industry segment             | Categorical         | ~6.3% null                                      |
| DOMINIO_EMAIL       | Registered email domain               | Categorical         | ~2.3% null                                      |
| PORTE               | Customer/company size                 | Categorical         | ~3.1% null                                      |
| CEP_2_DIG           | First two digits of the ZIP code      | Numeric/Categorical | ~0.08% null                                     |

Observations

* FLAG_PF shows a **high concentration of null values** and is a strong candidate for removal.
* DDD shows **format inconsistencies**, with values containing numbers and letters, in addition to a relevant number of nulls.
* The remaining variables show lower levels of missingness and can be treated during data preparation.
* ID_CLIENTE will be used as the key to relate this dataset to the other data sources.


In [0]:
cadastral_df = spark.table("credit_score.data.cadastral")
display(cadastral_df.limit(10))


In [0]:
num_rows = cadastral_df.count()
num_cols = len(cadastral_df.columns)
display(spark.createDataFrame([(num_rows, num_cols)], ["num_rows", "num_columns"]))


In [0]:
display(cadastral_df.dtypes)


In [0]:
display(cadastral_df.select("DDD").distinct())


In [0]:
display(cadastral_df.select("CEP_2_DIG").distinct())


In [0]:
from pyspark.sql.functions import col, count, when

total_count = cadastral_df.count()
missing_pct_df = (
    cadastral_df.select([
        (count(when(col(c).isNull(), c)) / total_count * 100).alias(c)
        for c in cadastral_df.columns
    ])
)
display(missing_pct_df)


# Info


The **info** dataset contains monthly information related to customers, allowing tracking of characteristics that may change over time. Each record represents a customer in a given reference period (SAFRA_REF).

The dataset has **24,401 records and 4 variables**.

Data Dictionary

| Variable           | Description                                              | Type     | Notes                                    |
| ------------------ | -------------------------------------------------------- | -------- | ---------------------------------------- |
| ID_CLIENTE         | Unique customer identifier                               | Integer  | Key for joining datasets                 |
| SAFRA_REF          | Reference period for the information                     | Date     | Represents the reference month           |
| RENDA_MES_ANTERIOR | Income recorded in the month before the reference period | Numeric  | ~2.94% null                              |
| NO_FUNCIONARIOS    | Number of customer employees                             | Numeric  | ~5.13% null                              |

Observations

* The dataset has a **temporal structure**, allowing analysis of how customer characteristics evolve over months.
* RENDA_MES_ANTERIOR has approximately **2.94% missing values**.
* NO_FUNCIONARIOS has approximately **5.13% missing values**.
* ID_CLIENTE and SAFRA_REF have no null values and will be important for temporal joins with the other datasets.
* Because this information changes over time, it is important to ensure that, when building features, only information **available up to the reference period** is used, avoiding *data leakage*.


In [0]:
info_df = spark.table("credit_score.data.info")
display(info_df.limit(10))


In [0]:
num_rows = info_df.count()
num_cols = len(info_df.columns)
display(spark.createDataFrame([(num_rows, num_cols)], ["num_rows", "num_columns"]))


In [0]:
display(info_df.dtypes)


In [0]:
from pyspark.sql.functions import col, count, when

total_count = info_df.count()
missing_pct_df = (
    info_df.select([
        (count(when(col(c).isNull(), c)) / total_count * 100).alias(c)
        for c in info_df.columns
    ])
)
display(missing_pct_df)


# Payments


The **pagamentos** dataset contains the history of billing charges and payments made by customers. Each record represents a billing charge associated with a customer and a reference period.

This is one of the main datasets for the project, as it contains **due dates and payment dates**, allowing identification of default according to the defined rule.

The dataset has **77,414 records and 8 variables**.

Data Dictionary

| Variable               | Description                               | Type     | Notes                             |
| ---------------------- | ----------------------------------------- | -------- | --------------------------------- |
| ID_CLIENTE             | Customer identifier                       | Integer  | Join key                          |
| SAFRA_REF              | Billing reference period                  | Date     | Enables temporal joins            |
| DATA_EMISSAO_DOCUMENTO | Billing issue date                        | Date     | No null values                    |
| DATA_PAGAMENTO         | Date the payment was made                 | Date     | No null values                    |
| DATA_VENCIMENTO        | Billing due date                          | Date     | No null values                    |
| VALOR_A_PAGAR          | Billing amount                            | Numeric  | ~1.51% null                       |
| TAXA                   | Fee associated with the billing charge    | Numeric  | No null values                    |
| ID_DOCUMENTO           | Unique document/billing charge identifier | Integer  | Document key                      |

Observations

* The dataset contains the information needed to **build the target variable**, comparing DATA_PAGAMENTO and DATA_VENCIMENTO.
* The rule used in the project considers a billing charge **in default when payment occurs 5 or more days late**.
* VALOR_A_PAGAR has approximately **1.51% missing values**, which must be handled during development.
* ID_DOCUMENTO identifies each billing charge individually and is also used when joining with other project information.
* The available dates allow building features related to **payment behavior and delay history**.


In [0]:
pagamentos_df = spark.table("credit_score.data.pagamentos")
display(pagamentos_df.limit(10))


In [0]:
num_rows = pagamentos_df.count()
num_cols = len(pagamentos_df.columns)
display(spark.createDataFrame([(num_rows, num_cols)], ["num_rows", "num_columns"]))


In [0]:
display(pagamentos_df.dtypes)


In [0]:
from pyspark.sql.functions import col, count, when

total_count = pagamentos_df.count()
missing_pct_df = (
    pagamentos_df.select([
        (count(when(col(c).isNull(), c)) / total_count * 100).alias(c)
        for c in pagamentos_df.columns
    ])
)
display(missing_pct_df)
